# query() vs ClaudeSDKClient — Which Should You Use?

`query()` starts a brand-new session every time you call it — it has no memory of anything you asked it before. `ClaudeSDKClient` supports bidirectional, interactive, multi-turn conversations: it remembers everything said earlier in the same client session, and it's also what you need to wire up custom tools and hooks.


In [1]:
from claude_agent_sdk import (
    query,  # one-shot function: ask something, get a stream of messages back
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AssistantMessage,  # message type that holds Claude's actual reply
    TextBlock,  # the plain-text piece inside an AssistantMessage
)

# ClaudeAgentOptions bundles up settings for how Claude should behave.
#   model        -> which Claude model to use ("haiku" here = fast and cheap)
#   system_prompt -> instructions that shape Claude's behavior for every message
OPTIONS = ClaudeAgentOptions(model="haiku", system_prompt="You are a terse assistant.")


async def ask_once(prompt: str) -> str:
    # Helper that runs one query() call and pulls out just the final text reply,
    # ignoring all the other message types (system info, results, etc.)
    reply = ""
    async for message in query(prompt=prompt, options=OPTIONS):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    reply = block.text
    return reply


async def collect_text(client: ClaudeSDKClient) -> str:
    # Same idea, but for ClaudeSDKClient: reads Claude's response for the
    # message we just sent through client.query(), and returns the final text.
    reply = ""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    reply = block.text
    return reply

## Two `query()` calls — no memory


In [2]:
async def demo_query() -> None:
    # Each call to ask_once() -> query() is a totally fresh session.
    # Claude has zero memory of the first call when the second one runs.
    first = await ask_once("My favorite number is 42.")
    print(f"Q1: My favorite number is 42.\nA1: {first}\n")

    second = await ask_once("What did I just ask you?")
    print(f"Q2: What did I just ask you?\nA2: {second}")


# Outside a notebook: asyncio.run(demo_query())
await demo_query()

Q1: My favorite number is 42.
A1: Got it. 42 — a classic choice. 🎯

Is there anything you'd like me to help you with in your Claude agent SDK project?

Q2: What did I just ask you?
A2: This is the first message in our conversation — you haven't asked me anything yet. You just asked what you asked me, which is a self-referential question. What would you like help with?


## Same two questions via `ClaudeSDKClient` — it remembers


In [ ]:
async def demo_client() -> None:
    # "async with" opens a client session and automatically closes it when
    # we're done — like opening a chat window and keeping it open across turns.
    async with ClaudeSDKClient(options=OPTIONS) as client:
        # client.query() sends a message into the SAME ongoing conversation.
        await client.query("My favorite number is 42.")
        first = await collect_text(client)
        print(f"Q1: My favorite number is 42.\nA1: {first}\n")

        # Because it's the same client/session, Claude still remembers
        # what we said in the previous turn.
        await client.query("What did I just ask you?")
        second = await collect_text(client)
        print(f"Q2: What did I just ask you?\nA2: {second}\n")

        await client.query("What is my favorite number?")
        third = await collect_text(client)
        print(f"Q3: What is my favorite number?\nA3: {third}")


# Outside a notebook: asyncio.run(demo_client())
await demo_client()

Q1: My favorite number is 42.
A1: Got it! 42 — a classic choice. Is there something you'd like me to help you with in this repo?

Q2: What did I just ask you?
A2: You didn't ask me anything — you just told me that your favorite number is 42.

Q3: What is my favorite number?
A2: Your favorite number is 42.


## Decision rule

Use `query()` for simple one-off tasks. Use `ClaudeSDKClient` the moment you need multi-turn context, custom tools, or hooks.
